In [6]:
import numpy as np
import matplotlib.pyplot as plt
import xtrack as xt
import sys
helpers_path = f'../' # specify the path for helper functions file
sys.path.insert(0, helpers_path)
from helpers_for_imperfections_model import install_orbit_correctors, add_correctors

In [7]:
# Load reference line
line = xt.Line.from_json(f"lattices/reference_lattice_LCC_V106/line_fccee_p_ring_LCC_106-2-3_z_merged_dipoles.json")
tt = line.get_table()

Loading line from dict: 100%|██████████| 14697/14697 [00:03<00:00, 4084.51it/s]


Done loading line from dict.           


### Add orbit correctors

In [8]:
# Filter out all quadrupoles
ttquad = tt.rows[tt.element_type == 'Quadrupole']

# Install orbit correctors at all the quadrupoles which do not already have an orbit corrector attached to them
hcor_names, vcor_names = install_orbit_correctors(line, ttquad.name)

print('Number of horizontal correctors:', len(hcor_names))
print('Number of vertical correctors:', len(vcor_names))

tt = line.get_table()
tthcor = tt.rows['hcor.*']
ttvcor = tt.rows['vcor.*']

# Double-checking the corrector installation
print('Checking the number of correctors in the line')
print('Number of horizontal correctors:', len(tthcor))
print('Number of vertical correctors:', len(ttvcor))

Slicing line: 100%|██████████| 21288/21288 [00:00<00:00, 115032.80it/s]


Number of horizontal correctors: 2679
Number of vertical correctors: 2679
Checking the number of correctors in the line
Number of horizontal correctors: 2679
Number of vertical correctors: 2679


### Add optics correctors

In [4]:
# Filter out normal quads
mask = [quad for quad in ttquad.name if line.element_dict[quad].k1s==0 and abs(line.element_dict[quad].k1)>0]
ttquadno = ttquad.rows[np.isin(ttquad.name, mask)]

# Filter out normal sexts
ttsext = tt.rows[tt.element_type=='Sextupole']
mask = [sext for sext in ttsext.name if line.element_dict[sext].k2s==0 and abs(line.element_dict[sext].k2)>0]
ttsextno = ttsext.rows[np.isin(ttsext.name, mask)]

# Add optics corrector trims
# normal quadrupole corrector trims to normal quadrupoles
add_correctors(line, ttquadno.name, type='normal', order=1, switch_name='on_qno_corrector')
# skew quadrupole corrector trims to normal sextupoles
add_correctors(line, ttsextno.name, type='skew', order=1, switch_name='on_qsk_corrector')

In [5]:
# Save the line with correctors
line.to_json(f"lattices/reference_lattice_LCC_V106/line_fccee_p_ring_LCC_106-2-3_z_merged_dipoles_with_correctors.json")

### Check the corrector installation

In [7]:
# Investigating the number of elements

tt = line.get_table()

ttdip = tt.rows[tt.element_type=='RBend']
ttquad = tt.rows[tt.element_type=='Quadrupole']
ttsext = tt.rows[tt.element_type=='Sextupole']
ttbpms = tt.rows['bpm.*']
tthcor = tt.rows['hcor.*']
ttvcor = tt.rows['vcor.*']

print('Number of dipoles:', len(ttdip))
print('Number of quadrupoles:', len(ttquad))
print('Number of sextupoles:', len(ttsext))
print('Number of BPMs:', len(ttbpms))
print('Number of H orbit correctors:', len(tthcor))
print('Number of V orbit correctors:', len(ttvcor))

Number of dipoles: 2432
Number of quadrupoles: 2679
Number of sextupoles: 1952
Number of BPMs: 2639
Number of H orbit correctors: 2679
Number of V orbit correctors: 2679
